# Router fine-tune: Qwen2.5-1.5B-Instruct + QLoRA

Fine-tunes a 1.5B open model to classify customer messages into `support | orders | escalation`, then compares **base vs fine-tuned vs a large API model** on a held-out test set (accuracy, latency, cost).

Runtime: **GPU (T4)** - Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Takes about 13 minutes end to end on a T4, 10 of them training. Cell 6 prints everything in `finetune/results/`, which is what gets committed back to the repository.

In [ ]:
# 1. Get the code and the libraries (safe to run again: it moves the copy to the latest commit)
import os
REPO = 'https://github.com/khizerabbas01/cadral-ai-ops-agent.git'
if not os.path.isdir('/content/repo'):
    !git clone -q $REPO /content/repo
%cd /content/repo
!git fetch -q origin main && git reset -q --hard FETCH_HEAD
!git log -1 --format='commit %h  %s'
!pip -q install -r finetune/requirements.txt
# Colab ships an old torchao that the installed peft refuses to load an adapter beside; nothing here uses torchao
!pip -q uninstall -y torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip list 2>/dev/null | grep -iE '^(torch|transformers|peft|datasets|accelerate|bitsandbytes) '

In [ ]:
# 2. Build the dataset (train/val/test with no template overlap between train and test)
!python finetune/build_dataset.py

In [ ]:
# 3. Fine-tune (QLoRA, rank 16, 3 epochs)
!python finetune/train.py --out finetune/out/router-lora

In [ ]:
# 4. Evaluate the base model and the fine-tuned model on the held-out test set
!python finetune/evaluate.py --adapter finetune/out/router-lora

In [ ]:
# 5. Optional: add the large API model's row. Needs a free Groq key saved as GROQ_API_KEY in Colab's Secrets panel;
#    without one this cell does nothing. The same command works from any machine, no GPU needed.
import os
from google.colab import userdata
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except Exception:
    print('No GROQ_API_KEY secret, skipped. Elsewhere: python finetune/evaluate.py --skip-local --api-model openai/gpt-oss-120b')
else:
    !python finetune/evaluate.py --skip-local --api-model openai/gpt-oss-120b

In [ ]:
# 6. Results: these three files are finetune/results/ - commit them to the repository
#    (the dashboard's Evaluations > Fine-tuning tab reads results.json)
import hashlib, pathlib
for name in ('results.md', 'train.json', 'results.json'):
    data = pathlib.Path('finetune/results', name).read_bytes()
    print(f'===== {name}  sha256={hashlib.sha256(data).hexdigest()} =====')
    print(data.decode())

In [ ]:
# 7. Optional: download the adapter to serve it behind CLASSIFIER_URL (finetune/serve.py)
DOWNLOAD = False  # set to True to download router-lora.zip (about 75 MB)
if DOWNLOAD:
    !zip -qr router-lora.zip finetune/out/router-lora finetune/results
    from google.colab import files
    files.download('router-lora.zip')